# label_fix_v1 생성 절차
원본 셀 17 → 19 → 20 → 21 → 22의 의존성을 정리했습니다. 초기 baseline과 supplement_v1 학습은 최종 모델 생성의 선행 조건이 아닙니다. 세 실험 모두 yolo11n.pt에서 시작합니다.

**이 노트북은 이번 작업에서 실행하지 않았으며 재학습은 기본 차단됩니다.** 원본·업로드 자료는 보존하고 데이터 준비는 별도 새 폴더에서만 수행합니다.

## 환경과 필요한 실제 파일
Python 환경: `/workspace/onboarding/venv/bin/python`, Ultralytics 8.4.182. `google.colab.files` 대신 명시적 로컬 경로를 사용합니다.

아직 없는 파일: **recycling_yolo_starter.zip** 및 학습 실행 시 필요한 사전학습 **yolo11n.pt**. 제공된 supplement ZIP과 패치 JSON만으로 starter 720장을 복원할 수 없습니다. 모델 ZIP의 배치 미리보기는 데이터셋이 아닙니다. 원래 GPU 학습 환경과 현재 CPU 환경은 다릅니다.

In [ ]:
from pathlib import Path
import json, hashlib, shutil, zipfile
import yaml
ASSETS = Path('/workspace/recycling_assets')
# 실제 파일을 지정하세요. starter ZIP은 아직 제공되지 않았습니다.
STARTER_ZIP = ASSETS / 'recycling_yolo_starter.zip'
SUPPLEMENT_ROOT = ASSETS / 'supplement/recycling_supplement_v1'
PATCH_JSON = ASSETS / 'recycling_label_patch_v1.json'
# 원본과 분리된 새 작업 폴더만 허용합니다. 기존 폴더는 덮어쓰지 않습니다.
WORK = ASSETS / 'prepared_label_fix_v1'
EXPECTED_COUNTS = {'train': 731, 'val': 90, 'test': 90}

## 1. 원본 데이터 복사 → train 11장 추가 → train 라벨 27개 수정
검증·테스트는 해시로 보존합니다. 실패한 준비 폴더는 자동 재사용하지 않으므로 원인을 해결한 뒤 새로운 WORK를 지정하세요.

In [ ]:
required = [STARTER_ZIP, SUPPLEMENT_ROOT / 'images/train',
            SUPPLEMENT_ROOT / 'labels/train', PATCH_JSON]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError('실제 파일이 필요합니다: ' + ', '.join(missing))
if WORK.exists():
    raise FileExistsError(f'새 WORK 경로를 지정하세요. 기존 자료 보존: {WORK}')
patch = json.loads(PATCH_JSON.read_text(encoding='utf-8'))
changes = patch['changes']
assert len(changes) == 27, '기록된 v1 패치는 27개입니다.'
paths = [item['path'] for item in changes]
assert len(set(paths)) == len(paths), '패치 경로 중복'
for rel in paths:
    p = Path(rel)
    assert not p.is_absolute() and len(p.parts) == 3
    assert p.parts[:2] == ('labels', 'train') and p.suffix == '.txt'
# ZIP 전체 경로를 검사한 뒤 새 폴더에만 풉니다.
with zipfile.ZipFile(STARTER_ZIP) as z:
    for info in z.infolist():
        target = (WORK / info.filename).resolve()
        assert target.is_relative_to(WORK.resolve()), 'ZIP 경로 이탈'
        assert (info.external_attr >> 16) & 0o170000 != 0o120000, 'ZIP 심볼릭 링크'
    assert 'recycling_yolo_starter/data.yaml' in z.namelist()
    WORK.mkdir(parents=True)
    z.extractall(WORK)
root = WORK / 'recycling_yolo_starter'
def split_hashes():
    return {str(p.relative_to(root)): hashlib.sha256(p.read_bytes()).hexdigest()
            for kind in ('images', 'labels') for split in ('val', 'test')
            for p in (root / kind / split).rglob('*') if p.is_file()}
unchanged = split_hashes()
assert unchanged, '검증/테스트 파일 없음'
images = sorted((SUPPLEMENT_ROOT / 'images/train').glob('*.jpg'))
assert len(images) == 11
for image in images:
    assert (SUPPLEMENT_ROOT / 'labels/train' / (image.stem + '.txt')).is_file()
# 충돌을 먼저 검사합니다. excluded/review의 사진은 학습에 넣지 않습니다.
copies = []
for image in images:
    for kind, src in [('images', image), ('labels', SUPPLEMENT_ROOT / 'labels/train' / (image.stem + '.txt'))]:
        target = root / kind / 'train' / src.name
        assert not target.exists(), f'추가 파일 충돌: {target}'
        copies.append((src, target))
for src, target in copies:
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, target)
# 원본 라벨이 정확히 before/after 중 하나인지 모두 확인한 뒤 수정합니다.
for item in changes:
    path = root / item['path']
    assert path.resolve().is_relative_to(root.resolve())
    assert path.read_text(encoding='utf-8').strip() in (item['before'].strip(), item['after'].strip()), str(path)
for item in changes:
    path = root / item['path']
    backup = root / 'label_backup_v1' / item['path']
    backup.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(path, backup)
    path.write_text(item['after'], encoding='utf-8')
for cache in (root / 'labels').rglob('*.cache'):
    cache.unlink()
assert split_hashes() == unchanged, 'val/test 변경 감지'
config = yaml.safe_load((root / 'data.yaml').read_text(encoding='utf-8'))
names = config['names']
classes = [names[i] for i in range(len(names))] if isinstance(names, dict) else names
assert classes == ['plastic', 'paper', 'metal'], classes
for split, expected in EXPECTED_COUNTS.items():
    imgs = [p for p in (root / 'images' / split).iterdir() if p.suffix.lower() in {'.jpg', '.jpeg', '.png'}]
    assert len(imgs) == expected, (split, len(imgs), expected)
    for img in imgs:
        label = root / 'labels' / split / (img.stem + '.txt')
        assert label.is_file(), str(label)
        for row in label.read_text(encoding='utf-8').splitlines():
            values = row.split()
            assert len(values) == 5, (label, row)
            cls, *coords = map(float, values)
            assert cls.is_integer() and 0 <= cls < 3
            assert all(0 <= v <= 1 for v in coords) and coords[2] > 0 and coords[3] > 0
    print(split, len(imgs))
config.update(path=str(root), train='images/train', val='images/val', test='images/test')
yaml_path = root / 'data.yaml'
yaml_path.write_text(yaml.safe_dump(config, allow_unicode=True), encoding='utf-8')
(WORK / 'preparation_history.json').write_text(json.dumps({
    'patch_sha256': hashlib.sha256(PATCH_JSON.read_bytes()).hexdigest(),
    'starter_sha256': hashlib.sha256(STARTER_ZIP.read_bytes()).hexdigest(),
    'supplement_files': {str(p.relative_to(SUPPLEMENT_ROOT)): hashlib.sha256(p.read_bytes()).hexdigest()
                         for p in SUPPLEMENT_ROOT.rglob('*') if p.is_file()},
    'patch': patch, 'val_test_sha256': unchanged,
}, ensure_ascii=False, indent=2), encoding='utf-8')
print('데이터 준비 완료. 학습은 실행하지 않았습니다:', yaml_path)

## 2. 최종 학습 — 기본 실행 차단
보존한 전체 설정은 `실험_기록/label_fix_v1_args.yaml`을 확인하세요. 아래는 원본의 명시적 설정을 유지합니다. 실험명 자동 증가를 피하며 실제 저장 경로를 사용합니다.

In [ ]:
# 안전장치: 전체 실행을 해도 재학습하지 않습니다.
RUN_TRAINING = False
if not RUN_TRAINING:
    raise RuntimeError('재학습 비활성화. 사용자가 별도로 학습을 결정한 경우에만 True로 변경하세요.')
import torch, ultralytics
from ultralytics import YOLO
assert ultralytics.__version__ == '8.4.182'
assert torch.cuda.is_available(), '원래 학습 설정은 CUDA device=0입니다. 현재 환경은 CPU입니다.'
assert yaml_path.is_file()
# 이전 best.pt에서 이어 학습하지 않고, 같은 사전학습 yolo11n.pt에서 독립 학습합니다.
PRETRAINED = ASSETS / 'yolo11n.pt'
assert PRETRAINED.is_file(), '원본 사전학습 yolo11n.pt를 준비하세요.'
PROJECT = ASSETS / 'new_training_runs'
assert not (PROJECT / 'label_fix_v1').exists(), '기존 결과를 보존하려면 새 PROJECT를 지정하세요.'
model = YOLO(str(PRETRAINED))
model.train(data=str(yaml_path), epochs=20, imgsz=640, batch=8,
            device=0, project=str(PROJECT), name='label_fix_v1', seed=42, exist_ok=False)
run_dir = Path(model.trainer.save_dir)
print('새 결과:', run_dir, 'best:', run_dir / 'weights/best.pt')

## 3. 새 학습 결과 내보내기 (선택)
이전 업로드 모델을 덮어쓰지 않습니다.

In [ ]:
if not RUN_TRAINING:
    raise RuntimeError('새 학습을 실행하지 않았습니다.')
archive_base = ASSETS / 'new_label_fix_run'
assert not archive_base.with_suffix('.zip').exists()
archive = shutil.make_archive(str(archive_base), 'zip', root_dir=run_dir)
print(archive)